<a href="https://colab.research.google.com/github/MsCrocus/mental-health-chatbot/blob/main/mhcb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --quiet \
    "langchain-core>=1.0.0" \
    "langchain-text-splitters>=1.0.0" \
    langchain-google-genai \
    langchain-community \
    sentence-transformers \
    "datasets>=2.0" \
    "faiss-cpu>=1.7" \
    "gradio>=4.0"

# --- 1. Kütüphanelerin ve API Anahtarının Hazırlanması ---
import os
import gradio as gr
import google.generativeai as genai
from google.colab import userdata
from datasets import load_dataset

# LangChain bileşenleri
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- 2. Colab Secrets'tan API Anahtarını Alma ve Yapılandırma ---
print("Google Colab Secrets'tan Gemini API anahtarına erişiliyor...")
try:
    api_key = userdata.get('GEMINI_API_KEY')
    if not api_key:
        raise ValueError("API anahtarı boş.")

    # API anahtarını doğrudan genai kütüphanesine tanıtıyoruz.
    genai.configure(api_key=api_key)

    print("API anahtarı başarıyla alındı ve yapılandırıldı.")
except Exception as e:

    print(f"Bir hata oluştu: {e}")
    raise ValueError("Colab Secrets'tan 'GEMINI_API_KEY' alınamadı! Lütfen sol menüdeki 'Anahtar' (🔑) simgesine tıklayarak anahtarınızı eklediğinizden emin olun.")

# --- 3. Veri Yükleme ve Hazırlama ---
print("Veri seti yükleniyor...")
# HuggingFaceDatasetLoader kullanarak veriyi doğrudan Document formatında yüklüyoruz.
loader = HuggingFaceDatasetLoader("aneerajsk/medchat_mental", page_content_column="text")
docs = loader.load()
print(f"{len(docs)} adet doküman yüklendi.")

# Metinleri daha küçük parçalara ayırmak için splitter'ı hazırlıyoruz.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = text_splitter.split_documents(docs)
print(f"Dokümanlar {len(splits)} parçaya ayrıldı.")

# --- 4. Vektör Veritabanı ve RAG Zinciri Oluşturma ---
print("Gömme modeli (embedding model) ve vektör veritabanı oluşturuluyor...")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# FAISS, bu vektörleri hafızada hızlı bir şekilde saklar ve arar.
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

# Retriever, kullanıcı sorgusuna en uygun metin parçalarını veritabanından bulur.
retriever = vectorstore.as_retriever()

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=api_key, temperature=0.7)


# Sisteme nasıl davranacağını söyleyen prompt şablonunu oluşturuyoruz.
SYSTEM_PROMPT = """
Sen, derinlemesine şefkatli ve anlayışlı bir zihin sağlığı asistanısın.
Birincil amacın, kullanıcının kendini duyulmuş, anlaşılmış ve desteklenmiş hissetmesini sağlamaktır.
Görevin, kullanıcının sorularını sana sağlanan bağlamı kullanarak, ancak bunu asla mekanik veya robotik bir şekilde yapmadan yanıtlamaktır.
Tonun her zaman sıcak, yargılayıcı olmayan ve cesaret verici olmalı.

ÖNEMLİ KURALLAR:
1.  **Sen bir terapist veya doktor değilsin.** Tıbbi tavsiye, teşhis veya tedavi sunma. Görevin dinlemek, destek olmak ve bağlamdaki bilgileri şefkatle aktarmaktır.
2.  **Bağlamı Şefkatle Çerçevele:** Cevaplarını sağlanan metin parçalarına (belgelere) dayandır, ancak bilgiyi *kopyalayıp yapıştırma*. Bilgiyi al ve kendi şefkatli sözlerinle *yeniden ifade et* veya *çerçevele*. Bağlamda olmayan bir bilgi ekleme, ancak bağlamdaki bilgiyi sunuş şeklin empatik olsun.
3.  **Duygusal Doğrulama Esastır:** Yanıtına başlarken, eğer kullanıcı bir duygu (üzüntü, kaygı, stres, vb.) paylaşıyorsa, önce bu duyguyu fark et ve doğrula.
    * Örneğin: "Bu durumun sizin için ne kadar zorlayıcı olduğunu duyabiliyorum..." veya "Böyle hissetmeniz çok anlaşılır..."
4.  **Yüzeysellikten Kaçın:** "Kısa" yanıtlar yerine "anlamlı" ve "destekleyici" yanıtlar ver. Yanıtların aceleci veya eksik hissettirmesin. Konuyu net bir şekilde ele alırken samimiyeti koru.
5.  **Acil Durum Yönlendirmesi:** Eğer kullanıcı acil bir krizde gibi görünüyorsa (kendine veya başkalarına zarar verme düşünceleri gibi), doğrudan profesyonel yardım aramalarını şiddetle tavsiye et.
    * Örneğin: "Bu düşüncelerle yalnız başınıza mücadele etmek zorunda olmadığınızı bilmenizi isterim. Acil yardıma ihtiyacınız varsa, lütfen yerel acil durum numaranızı arayın veya bir zihin sağlığı uzmanıyla konuşun."

YANIT AKIŞI:
1.  **Doğrula:** Kullanıcının duygusunu anladığını göster.
2.  **Yanıtla:** Soruyu cevaplamak için BAĞLAM'daki ilgili bilgiyi şefkatli bir dille sun.
3.  **Destekle:** Konuşmayı destekleyici veya cesaret verici bir cümle ile bitir.

BAĞLAM:
{context}

SORU: {question}

CEVAP:
"""

prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)

# Tüm bileşenleri bir RAG zincirinde birleştiriyoruz.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG zinciri hazır!")

# --- 5. Gradio Arayüzünü Oluşturma ---
def chat_function(message, history):
    if not message or not message.strip():
        return "Lütfen bir soru sorun."
    try:
        response = rag_chain.invoke(message)
        return response
    except Exception as e:
        return f"Bir hata oluştu: {e}"

demo = gr.ChatInterface(
    fn=chat_function,
    title="Zihin Sağlığı Destek Asistanı",
    description="Zihin sağlığı ile ilgili sorularınızı sorun. Bu asistan, size destek olmak için burada. Unutmayın, bu bir tıbbi tavsiye değildir.",
    examples=[
        "Gelecek kaygımı nasıl yönetebilirim?",
        "Son zamanlarda hiçbir şeyden keyif almıyorum. Motivasyonumu nasıl geri kazanabilirim?",
        "Kendimi sakinleştirmek için şu anda yapabileceğim basit bir nefes egzersizi var mı?",
        "'Öz-şefkat nedir ve kendime karşı nasıl daha anlayışlı olabilirim?"
    ],
    theme="soft",
    chatbot=gr.Chatbot(height=500),
)

# --- 6. Uygulamayı Başlatma ---
print("Gradio arayüzü başlatılıyor...")
demo.launch(share=True)